# Activation Steering

**Activation steering** adds or removes direction vectors from a model's internal activations during the forward pass. This lets you:

- Push the model toward or away from specific behaviors
- Study which directions in activation space correspond to which behaviors
- Compare before/after steering effects

This notebook covers the `irtk.steering` module.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

import irtk
from irtk import steering

# Load GPT-2 Small
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## 1. Computing a Steering Vector

The most common approach: take the mean activation difference between contrasting prompts.
For example, positive vs negative sentiment prompts.

In [ ]:
# Contrasting prompts
positive_prompts = [
    "I love this movie, it's absolutely",
    "This restaurant is amazing and the food is",
    "What a wonderful day, I feel so",
]
negative_prompts = [
    "I hate this movie, it's absolutely",
    "This restaurant is terrible and the food is",
    "What an awful day, I feel so",
]

pos_tokens = [model.to_tokens(p) for p in positive_prompts]
neg_tokens = [model.to_tokens(p) for p in negative_prompts]

# Compute steering vector from layer 10's residual stream
hook_name = "blocks.10.hook_resid_post"
steer_vec = steering.compute_steering_vector(
    model, pos_tokens, neg_tokens, hook_name
)
print(f"Steering vector norm: {np.linalg.norm(steer_vec):.4f}")
print(f"Steering vector shape: {steer_vec.shape}")

## 2. Applying the Steering Vector

Use `add_vector` to inject the direction during a forward pass.

In [ ]:
prompt = "I thought the performance was"
tokens = model.to_tokens(prompt)

# Baseline prediction
baseline_logits = model(tokens)
baseline_probs = jax.nn.softmax(baseline_logits[-1])

# Steered prediction (positive direction)
steer_vec_jax = jnp.array(steer_vec)
steered_logits = steering.add_vector(
    model, tokens, hook_name, steer_vec_jax, alpha=5.0
)
steered_probs = jax.nn.softmax(steered_logits[-1])

# Compare top predictions
print("Baseline top 5:")
for idx in jnp.argsort(baseline_probs)[::-1][:5]:
    print(f"  {tokenizer.decode([int(idx)]):>15s}: {float(baseline_probs[idx]):.1%}")

print("\nSteered (positive) top 5:")
for idx in jnp.argsort(steered_probs)[::-1][:5]:
    print(f"  {tokenizer.decode([int(idx)]):>15s}: {float(steered_probs[idx]):.1%}")

## 3. Steering in the Negative Direction

Use negative alpha to steer away from the positive direction.

In [ ]:
neg_steered_logits = steering.add_vector(
    model, tokens, hook_name, steer_vec_jax, alpha=-5.0
)
neg_steered_probs = jax.nn.softmax(neg_steered_logits[-1])

print("Steered (negative) top 5:")
for idx in jnp.argsort(neg_steered_probs)[::-1][:5]:
    print(f"  {tokenizer.decode([int(idx)]):>15s}: {float(neg_steered_probs[idx]):.1%}")

## 4. Removing a Direction with `subtract_vector`

This projects out a direction rather than adding it. Useful for ablating specific features.

In [ ]:
removed_logits = steering.subtract_vector(
    model, tokens, hook_name, steer_vec_jax, alpha=1.0
)
removed_probs = jax.nn.softmax(removed_logits[-1])

print("Direction removed top 5:")
for idx in jnp.argsort(removed_probs)[::-1][:5]:
    print(f"  {tokenizer.decode([int(idx)]):>15s}: {float(removed_probs[idx]):.1%}")

## 5. Steered Generation

Apply steering during autoregressive generation to produce text with modified behavior.

In [ ]:
prompt = "The weather today is"
tokens = model.to_tokens(prompt)

# Generate without steering
from irtk.generation import generate
baseline_gen = generate(model, tokens, max_new_tokens=20, temperature=0.0)
print(f"Baseline: {tokenizer.decode(np.array(baseline_gen))}")

# Generate with positive steering
pos_gen = steering.steer_generation(
    model, tokens, hook_name, steer_vec_jax,
    alpha=3.0, max_new_tokens=20, temperature=0.0,
)
print(f"Positive: {tokenizer.decode(np.array(pos_gen))}")

# Generate with negative steering
neg_gen = steering.steer_generation(
    model, tokens, hook_name, steer_vec_jax,
    alpha=-3.0, max_new_tokens=20, temperature=0.0,
)
print(f"Negative: {tokenizer.decode(np.array(neg_gen))}")

## 6. Activation Diff from Single Pairs

For quick exploration, compute the steering vector from a single pair of prompts.

In [ ]:
tokens_a = model.to_tokens("I love this, it's great")
tokens_b = model.to_tokens("I hate this, it's awful")

diff_vec = steering.activation_diff_at_hook(
    model, tokens_a, tokens_b, hook_name
)
print(f"Single-pair diff norm: {np.linalg.norm(diff_vec):.4f}")
print(f"Cosine similarity with multi-pair vector: "
      f"{np.dot(diff_vec, steer_vec) / (np.linalg.norm(diff_vec) * np.linalg.norm(steer_vec) + 1e-10):.4f}")

## Summary

| Function | Purpose |
|----------|--------|
| `compute_steering_vector()` | Mean diff between contrasting prompt activations |
| `add_vector()` | Add a direction during forward pass |
| `subtract_vector()` | Project out a direction (remove feature) |
| `steer_generation()` | Generate text with steering applied |
| `activation_diff_at_hook()` | Quick single-pair diff vector |